In [3]:
import pandas as pd
import pickle
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
movies=pd.read_csv("tmdb_5000_movies.csv")
credits=pd.read_csv("tmdb_5000_credits.csv")

In [5]:
movies.head()
credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [6]:
print(movies.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [7]:
print(credits.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4803 non-null   int64 
 1   title     4803 non-null   object
 2   cast      4803 non-null   object
 3   crew      4803 non-null   object
dtypes: int64(1), object(3)
memory usage: 150.2+ KB
None


In [8]:
merged=movies.merge(credits,on="title")
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [9]:
print(merged.shape)

(4809, 23)


In [10]:
merged=merged[['movie_id','title','genres','overview','keywords','vote_average','cast','crew']]
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   movie_id      4809 non-null   int64  
 1   title         4809 non-null   object 
 2   genres        4809 non-null   object 
 3   overview      4806 non-null   object 
 4   keywords      4809 non-null   object 
 5   vote_average  4809 non-null   float64
 6   cast          4809 non-null   object 
 7   crew          4809 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 300.7+ KB


In [11]:
merged['overview']=merged['overview'].fillna('')

In [12]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   movie_id      4809 non-null   int64  
 1   title         4809 non-null   object 
 2   genres        4809 non-null   object 
 3   overview      4809 non-null   object 
 4   keywords      4809 non-null   object 
 5   vote_average  4809 non-null   float64
 6   cast          4809 non-null   object 
 7   crew          4809 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 300.7+ KB


In [13]:
merged.shape

(4809, 8)

In [14]:
merged['genres']

0       [{"id": 28, "name": "Action"}, {"id": 12, "nam...
1       [{"id": 12, "name": "Adventure"}, {"id": 14, "...
2       [{"id": 28, "name": "Action"}, {"id": 12, "nam...
3       [{"id": 28, "name": "Action"}, {"id": 80, "nam...
4       [{"id": 28, "name": "Action"}, {"id": 12, "nam...
                              ...                        
4804    [{"id": 28, "name": "Action"}, {"id": 80, "nam...
4805    [{"id": 35, "name": "Comedy"}, {"id": 10749, "...
4806    [{"id": 35, "name": "Comedy"}, {"id": 18, "nam...
4807                                                   []
4808                  [{"id": 99, "name": "Documentary"}]
Name: genres, Length: 4809, dtype: object

In [15]:
def convert(text):
    l=[]
    for i in ast.literal_eval(text):
        l.append(i['name'])
    return l

In [16]:
merged['genres']=merged['genres'].apply(convert)
merged['keywords']=merged['keywords'].apply(convert)

In [17]:
merged['genres'].head()
merged['keywords'].head()

0    [culture clash, future, space war, space colon...
1    [ocean, drug abuse, exotic island, east india ...
2    [spy, based on novel, secret agent, sequel, mi...
3    [dc comics, crime fighter, terrorist, secret i...
4    [based on novel, mars, medallion, space travel...
Name: keywords, dtype: object

In [18]:
def top3_cast(text):
    c=[]
    count=0
    for i in ast.literal_eval(text):
        if count<3:
            c.append(i['name'])
            count+=1
        else:
            break
    return c

In [19]:
merged['cast']=merged['cast'].apply(top3_cast)
merged['cast'].head()

0    [Sam Worthington, Zoe Saldana, Sigourney Weaver]
1       [Johnny Depp, Orlando Bloom, Keira Knightley]
2        [Daniel Craig, Christoph Waltz, Léa Seydoux]
3        [Christian Bale, Michael Caine, Gary Oldman]
4      [Taylor Kitsch, Lynn Collins, Samantha Morton]
Name: cast, dtype: object

In [20]:
def director(text):
    d=[]
    for i in ast.literal_eval(text):
        if(i['job']=='Director'):
            d.append(i['name'])
    return d

In [21]:
merged['crew']=merged['crew'].apply(director)
merged['crew'].head()

0        [James Cameron]
1       [Gore Verbinski]
2           [Sam Mendes]
3    [Christopher Nolan]
4       [Andrew Stanton]
Name: crew, dtype: object

In [22]:
def extract(text):
    try:
        return text.split()
    except:
        return []

In [23]:
merged['overview']=merged['overview'].apply(extract)
merged['overview'].head()

0    [In, the, 22nd, century,, a, paraplegic, Marin...
1    [Captain, Barbossa,, long, believed, to, be, d...
2    [A, cryptic, message, from, Bond’s, past, send...
3    [Following, the, death, of, District, Attorney...
4    [John, Carter, is, a, war-weary,, former, mili...
Name: overview, dtype: object

In [24]:
def spaces(text):
    result=[]
    for i in text:
        result.append(i.replace(" ",""))
    return result

In [25]:
merged['genres']=merged['genres'].apply(spaces)
merged['keywords']=merged['keywords'].apply(spaces)
merged['cast']=merged['cast'].apply(spaces)
merged['crew']=merged['crew'].apply(spaces)
merged['crew'].head()

0        [JamesCameron]
1       [GoreVerbinski]
2           [SamMendes]
3    [ChristopherNolan]
4       [AndrewStanton]
Name: crew, dtype: object

In [26]:
merged['tags']=merged['genres'] + merged['overview']+merged['keywords']+merged['cast']+merged['crew']

In [27]:
merged['tags'].head()

0    [Action, Adventure, Fantasy, ScienceFiction, I...
1    [Adventure, Fantasy, Action, Captain, Barbossa...
2    [Action, Adventure, Crime, A, cryptic, message...
3    [Action, Crime, Drama, Thriller, Following, th...
4    [Action, Adventure, ScienceFiction, John, Cart...
Name: tags, dtype: object

In [28]:
new_df=merged[['movie_id','title','vote_average','tags']]

In [29]:
new_df.head()

,movie_id,title,vote_average,tags
0,19995,Avatar,7.2,"[Action, Adventure, Fantasy, ScienceFiction, I..."
1,285,Pirates of the Caribbean: At World's End,6.9,"[Adventure, Fantasy, Action, Captain, Barbossa..."
2,206647,Spectre,6.3,"[Action, Adventure, Crime, A, cryptic, message..."
3,49026,The Dark Knight Rises,7.6,"[Action, Crime, Drama, Thriller, Following, th..."
4,49529,John Carter,6.1,"[Action, Adventure, ScienceFiction, John, Cart..."


In [30]:
def list_to_text(text):
    return " ".join(text)

In [31]:
new_df['tags']=new_df['tags'].apply(list_to_text)
new_df['tags'].head()

/var/folders/27/nj75n8fn3fz7pydl39prbxhh0000gn/T/ipykernel_37425/969439916.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(list_to_text)


0    Action Adventure Fantasy ScienceFiction In the...
1    Adventure Fantasy Action Captain Barbossa, lon...
2    Action Adventure Crime A cryptic message from ...
3    Action Crime Drama Thriller Following the deat...
4    Action Adventure ScienceFiction John Carter is...
Name: tags, dtype: object

In [32]:
new_df['tags'] = new_df['tags'].apply(str.lower)

/var/folders/27/nj75n8fn3fz7pydl39prbxhh0000gn/T/ipykernel_37425/266116863.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(str.lower)


In [33]:
cv=CountVectorizer(max_features=5000,stop_words='english')
vectors=cv.fit_transform(new_df['tags']).toarray()

In [34]:
similar=cosine_similarity(vectors)

In [ ]:
pickle.dump(new_df,open("movies.pkl","wb"))
pickle.dump(similar,open("similar.pkl",'wb'))